# Install Libraries

In [ ]:
%pip install numpy matplotlib

# Tic-Tac-Toe — Tabular Q-Learning with Self-Play

## The Game
Tic-Tac-Toe is played on a 3×3 grid. Two players alternate placing their mark (X or O) until one achieves three in a row or the board is full (draw).

```
 0 | 1 | 2
---+---+---
 3 | 4 | 5
---+---+---
 6 | 7 | 8
```

## RL Formulation
- **State**: tuple of 9 values, each 0 (empty), 1 (X/agent), or -1 (O/opponent)
- **Action**: index 0–8 of an empty cell
- **Reward**: +1 win, -1 loss, 0 draw/step
- **Opponent**: plays random legal moves

## Why Tabular Q-Learning Works Here
The full Tic-Tac-Toe state space has 5,478 legal positions — small enough to enumerate with a dictionary. Each Q-value is stored keyed by `(state_tuple, action)`.

# Create & Test Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle, os
from collections import defaultdict


class TicTacToeEnv:
    """Minimal Tic-Tac-Toe environment. Agent plays X (1), opponent plays O (-1)."""

    WINS = [
        (0,1,2), (3,4,5), (6,7,8),
        (0,3,6), (1,4,7), (2,5,8),
        (0,4,8), (2,4,6),
    ]

    def reset(self):
        self.board = [0] * 9
        return tuple(self.board)

    def _check_winner(self, player):
        return any(all(self.board[i] == player for i in combo) for combo in self.WINS)

    def _legal_actions(self):
        return [i for i, v in enumerate(self.board) if v == 0]

    def step(self, action):
        assert self.board[action] == 0, 'Illegal move'
        self.board[action] = 1
        if self._check_winner(1):
            return tuple(self.board), +1.0, True
        legal = self._legal_actions()
        if not legal:
            return tuple(self.board), 0.0, True
        opp = np.random.choice(legal)
        self.board[opp] = -1
        if self._check_winner(-1):
            return tuple(self.board), -1.0, True
        if not self._legal_actions():
            return tuple(self.board), 0.0, True
        return tuple(self.board), 0.0, False

    def render(self, board=None):
        b = board or self.board
        sym = {0: '.', 1: 'X', -1: 'O'}
        for r in range(3):
            print(' | '.join(sym[b[r*3+c]] for c in range(3)))
            if r < 2:
                print('--+---+--')
        print()


# Smoke test
env = TicTacToeEnv()
state = env.reset()
done = False
while not done:
    legal = [i for i, v in enumerate(state) if v == 0]
    action = np.random.choice(legal)
    state, reward, done = env.step(action)
env.render()
print('Game result reward:', reward)

# Train Agent

In [ ]:
N_EPISODES    = 80_000
ALPHA         = 0.5
GAMMA         = 0.95
EPSILON_START = 1.0
EPSILON_END   = 0.05
EPSILON_DECAY = 0.99993

CHECKPOINT_EPISODES = [0, N_EPISODES // 4, N_EPISODES // 2, 3 * N_EPISODES // 4, N_EPISODES - 1]
CHECKPOINT_LABELS   = ['Untrained (ep 0)', '25%', '50%', '75%', '100% (fully trained)']
os.makedirs('models/tictactoe', exist_ok=True)

Q = defaultdict(float)

def get_q(state, action):
    return Q[(state, action)]

def best_action(state):
    legal = [i for i, v in enumerate(state) if v == 0]
    return max(legal, key=lambda a: get_q(state, a))

env = TicTacToeEnv()
epsilon = EPSILON_START
results = []

for episode in range(N_EPISODES):
    if episode in CHECKPOINT_EPISODES:
        idx = CHECKPOINT_EPISODES.index(episode)
        path = f'models/tictactoe/qtable_ep{episode}.pkl'
        with open(path, 'wb') as f:
            pickle.dump(dict(Q), f)
        print(f'Saved checkpoint [{CHECKPOINT_LABELS[idx]}] -> {path}')

    state = env.reset()
    while True:
        legal = [i for i, v in enumerate(state) if v == 0]
        action = np.random.choice(legal) if np.random.random() < epsilon else best_action(state)
        next_state, reward, done = env.step(action)

        if done:
            Q[(state, action)] += ALPHA * (reward - get_q(state, action))
            results.append(int(reward))
            break
        else:
            next_legal = [i for i, v in enumerate(next_state) if v == 0]
            future = max(get_q(next_state, a) for a in next_legal) if next_legal else 0.0
            Q[(state, action)] += ALPHA * (reward + GAMMA * future - get_q(state, action))
            state = next_state

    epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)

    if (episode + 1) % 10_000 == 0:
        last = results[-10_000:]
        print(f'Ep {episode+1:6d} | ε={epsilon:.3f} | '
              f'Win={last.count(1)/len(last):.2%} '
              f'Draw={last.count(0)/len(last):.2%} '
              f'Loss={last.count(-1)/len(last):.2%}')

# Save final
with open(f'models/tictactoe/qtable_ep{N_EPISODES-1}.pkl', 'wb') as f:
    pickle.dump(dict(Q), f)
print(f'Q-table entries: {len(Q)}')

# Evaluate & Visualize

In [ ]:
window = 2000
wins_ma   = np.convolve([1 if r==1  else 0 for r in results], np.ones(window)/window, mode='valid')
draws_ma  = np.convolve([1 if r==0  else 0 for r in results], np.ones(window)/window, mode='valid')
losses_ma = np.convolve([1 if r==-1 else 0 for r in results], np.ones(window)/window, mode='valid')

plt.figure(figsize=(10, 4))
plt.plot(wins_ma,   label='Win',  color='green')
plt.plot(draws_ma,  label='Draw', color='orange')
plt.plot(losses_ma, label='Loss', color='red')
for ep, label in zip(CHECKPOINT_EPISODES[1:], CHECKPOINT_LABELS[1:]):
    x = min(ep, len(wins_ma) - 1)
    plt.axvline(x, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('Episode')
plt.ylabel(f'Rate (moving avg window={window})')
plt.title('Tic-Tac-Toe — Q-Learning Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Compare: Untrained vs Intermediate vs Fully Trained

Load each saved Q-table and evaluate its greedy win/draw/loss rate over 2000 games against a random opponent.

In [ ]:
def eval_qtable_ttt(q_dict, n_episodes=2000):
    env = TicTacToeEnv()
    wins, draws, losses = 0, 0, 0
    for _ in range(n_episodes):
        state = env.reset()
        while True:
            legal = [i for i, v in enumerate(state) if v == 0]
            action = max(legal, key=lambda a: q_dict.get((state, a), 0.0))
            state, reward, done = env.step(action)
            if done:
                if reward > 0:   wins   += 1
                elif reward < 0: losses += 1
                else:            draws  += 1
                break
    return wins/n_episodes, draws/n_episodes, losses/n_episodes

checkpoint_paths = [f'models/tictactoe/qtable_ep{ep}.pkl' for ep in CHECKPOINT_EPISODES]
wr_wins, wr_draws, wr_losses = [], [], []

print(f'{"Checkpoint":30s}  Win     Draw    Loss')
print('-' * 55)
for path, label in zip(checkpoint_paths, CHECKPOINT_LABELS):
    with open(path, 'rb') as f:
        q = pickle.load(f)
    w, d, l = eval_qtable_ttt(q)
    wr_wins.append(w); wr_draws.append(d); wr_losses.append(l)
    print(f'{label:30s}  {w:.3f}   {d:.3f}   {l:.3f}')

# Stacked bar chart
x = np.arange(len(CHECKPOINT_LABELS))
fig, ax = plt.subplots(figsize=(9, 4))
p1 = ax.bar(x, wr_wins,   label='Win',  color='#2ca02c')
p2 = ax.bar(x, wr_draws,  bottom=wr_wins, label='Draw', color='#ff7f0e')
p3 = ax.bar(x, wr_losses, bottom=np.array(wr_wins)+np.array(wr_draws), label='Loss', color='#d62728')

for bars, vals in [(p1, wr_wins), (p2, wr_draws), (p3, wr_losses)]:
    for bar, v in zip(bars, vals):
        if v > 0.04:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_y() + bar.get_height()/2,
                    f'{v:.2f}', ha='center', va='center', color='white', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(CHECKPOINT_LABELS, rotation=15, ha='right')
ax.set_ylim(0, 1)
ax.set_ylabel('Rate (2000 games vs random)')
ax.set_title('Tic-Tac-Toe — Win/Draw/Loss at Each Training Checkpoint')
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Show a sample game for untrained vs fully trained
def play_game(q_dict, label):
    print(f'=== {label} ===')
    env = TicTacToeEnv()
    state = env.reset()
    done = False
    step = 0
    while not done:
        legal = [i for i, v in enumerate(state) if v == 0]
        action = max(legal, key=lambda a: q_dict.get((state, a), 0.0))
        state, reward, done = env.step(action)
        step += 1
        print(f'After move {step}:')
        env.render(list(state))
    outcome = {1: 'X wins!', -1: 'O wins!', 0: 'Draw!'}
    print(outcome[int(reward)])

for path, label in [(checkpoint_paths[0], CHECKPOINT_LABELS[0]),
                    (checkpoint_paths[-1], CHECKPOINT_LABELS[-1])]:
    with open(path, 'rb') as f:
        q = pickle.load(f)
    play_game(q, label)
    print()